# 00 — Subset OISST v2.1 to the Australian domain and climate-index boxes

Reads the yearly OISST files from ia39, writes
- `oisst_aus_daily.zarr` — daily SST over `cfg.DOMAIN`, chunked `time=-1` for MHW detection
- `index_boxes_daily.nc` — daily area-weighted SST in the Niño3.4 and DMI boxes

Kernel: **Python (mhw3d)**. ARE storage: `gdata/ia39+gdata/xp65+scratch/m35`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
print("OISST:", cfg.OISST_DIR)
print("Output:", cfg.OUT)

In [ ]:
client = cfg.start_client()
client

In [ ]:
files = cfg.oisst_files()
ds = xr.open_mfdataset(files, preprocess=cfg.preprocess_oisst, combine="by_coords",
                       parallel=True, chunks={"time": 366, "lat": 720, "lon": 720})
ds

## Time-axis check

In [ ]:
t = ds.time.to_index()
dt = np.diff(t.values).astype("timedelta64[D]").astype(int)
print(f"{t[0].date()} → {t[-1].date()}, {len(t)} days")
print("non-daily steps:", [(t[i].date(), t[i + 1].date()) for i in np.where(dt != 1)[0]])
assert t.is_unique and t.is_monotonic_increasing

## Domain subset → zarr

In [ ]:
aus = cfg.sel_box(ds.sst, cfg.DOMAIN).astype("float32")
aus.encoding = {}
aus = aus.chunk({"time": -1, **cfg.CHUNKS_SPACE})
aus.attrs.update(source=str(cfg.OISST_DIR), note="OISST v2.1 AVHRR-only, subset to DOMAIN; time floored to 00:00")
aus.to_dataset(name="sst").to_zarr(cfg.F_SST, mode="w", consolidated=True)

## Index-box means

In [ ]:
idx = xr.Dataset({k: cfg.box_mean(ds.sst, b) for k, b in cfg.INDEX_BOXES.items()}).compute()
idx.attrs["boxes"] = str(cfg.INDEX_BOXES)
idx.to_netcdf(cfg.F_INDICES_DAILY)
idx.to_array("box").plot.line(x="time", hue="box", lw=0.4, figsize=(10, 3));

## Sanity check

In [ ]:
sst = xr.open_zarr(cfg.F_SST).sst
print(sst)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sst.sel(time="2011-02-15").plot(ax=ax[0], cmap="turbo")
sst.sel(lat=-32.0, lon=115.0, method="nearest").sel(time=slice("2010", "2012")).plot(ax=ax[1])
ax[1].set_title("~32°S 115°E (WA 2011 MHW)");